# Implement Beam Search for LLM Decoding

**Difficulty**: 🟡 Medium

### Problem Statement

Beam search is a deterministic decoding algorithm that maintains multiple candidate sequences (beams) at each time step. Instead of greedily picking the single best token, beam search expands each beam by the entire vocabulary, scores all candidates by cumulative log-probability, and keeps only the top `beam_width` sequences. This explores a broader search space and often produces higher-quality outputs than greedy decoding.

Your goal is to implement beam search from scratch — including the two things every real implementation needs: **EOS handling** (a beam that emits the end-of-sequence token is finished) and **length normalisation** (otherwise the sum of log-probs, which only ever decreases, systematically prefers short outputs).

For testing we use a dummy *bigram* language model: `model(token_id)` returns next-token logits given only the last token. With a real transformer you would feed each beam's whole prefix — or just its last token plus that beam's KV cache, which then has to be reordered with the surviving beam indices after every step.

---

### Requirements

1. **Initialization**
   - Start with a single beam containing just the `start_token`.
   - Initialize its cumulative log-probability to 0.0.
   - (Starting from `beam_width` copies of the same beam would make every copy pick the same best token at step 1 — `beam_width` identical beams.)

2. **Expansion**
   - At each step, feed each beam's last token into the model to get logits.
   - Convert logits to log-probabilities using `log_softmax`.
   - For each beam, create `vocab_size` candidate extensions, each with an updated cumulative score: `score + log p(token | beam)`.

3. **Pruning**
   - From all candidates across all beams, keep only the top `beam_width` by cumulative log-probability.

4. **EOS handling** (`eos_token`, optional)
   - A candidate that ends in `eos_token` is a *finished hypothesis*: set it aside and never extend it; the beam keeps `beam_width` live (unfinished) candidates.
   - You may stop early once `beam_width` hypotheses have finished.

5. **Output**
   - When the sequence reaches `max_len` (start token included), the finished hypotheses and the beams still alive compete together.
   - Rank them by the length-normalised score `cumulative_log_prob / num_generated_tokens ** length_penalty` (`length_penalty=0` is the raw sum; `1` is the average log-prob per token) and return the best sequence.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Use a dummy model that returns random logits for any input token.
- ✅ Without EOS the output sequence has length `max_len` (start token included); a finished hypothesis ends with `eos_token`.
- ✅ The returned sequence has the highest length-normalised score among the final hypotheses.

---

**Companies**: Google, DeepMind, Meta, Apple

---

<details>
  <summary>💡 Hint</summary>

  - Represent each beam as a tuple: `(sequence_list, cumulative_log_prob)`.
  - At each step, collect all candidates into a single list, sort by score, and take the top `beam_width`.
  - Use `F.log_softmax(logits, dim=-1)` to convert model output to log-probabilities.
  - Vectorise the ranking: stack the beams' log-probs into a `(num_beams, vocab_size)` matrix, add the running scores as a column, flatten and `topk`; then `beam = idx // vocab_size`, `token = idx % vocab_size`.
  - Take `2 * beam_width` candidates, so that even if some end in EOS there are still `beam_width` live ones.
  - The dummy model can simply be a `nn.Linear` or even just `torch.randn` — the algorithm is the same regardless.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Data generation / setup
torch.manual_seed(42)
vocab_size = 50257

# Dummy language model: given any token, returns random logits over the vocabulary.
# In practice this would be a real transformer, but for testing beam search
# the algorithm is the same.
class DummyLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed = nn.Embedding(vocab_size, 64)
        self.head = nn.Linear(64, vocab_size)
    
    def forward(self, token_id):
        """Takes a single token id (int) and returns logits of shape (1, vocab_size)."""
        x = self.embed(torch.tensor([token_id]))
        return self.head(x)

model = DummyLM(vocab_size)
start_token = 0  # Beginning-of-sequence token
print(f"Dummy model created with vocab_size={vocab_size}")
print(f"Sample output shape: {model(start_token).shape}")

In [ ]:
def beam_search(model, start_token, beam_width=5, max_len=20, eos_token=None, length_penalty=1.0):
    """
    Perform beam search decoding.
    
    Args:
        model: A language model that takes a token id and returns logits (1, vocab_size)
        start_token (int): The starting token id (counts towards max_len)
        beam_width (int): Number of beams to maintain
        max_len (int): Maximum sequence length to generate, start token included
        eos_token (int, optional): A beam that emits this token is finished and is never extended
        length_penalty (float): Final hypotheses are ranked by
            score / num_generated_tokens ** length_penalty (0 = raw sum of log-probs)
        
    Returns:
        list: The best sequence (list of token ids). Without EOS it has length max_len;
              a finished hypothesis ends with eos_token.
    """
    # Step 1: Initialize with a SINGLE beam. Starting from beam_width copies of
    # [start_token] would make every beam pick the same best token at step 1.
    beams = [[start_token]]          # live (unfinished) sequences
    scores = torch.zeros(1)          # cumulative log-prob of each live sequence
    finished = []                    # (sequence, cumulative log-prob) of beams that emitted EOS
    
    # Step 2: Iterate for max_len - 1 steps (start_token is already the first token)
    for step in range(max_len - 1):
        # Expand: next-token log-probs for every live beam -> (num_beams, vocab_size)
        with torch.no_grad():
            log_probs = torch.cat([F.log_softmax(model(seq[-1]), dim=-1) for seq in beams])
        vocab_size = log_probs.size(-1)
        
        # Running total of every (beam, token) candidate, flattened to one ranking
        cand_scores = (scores.unsqueeze(1) + log_probs).view(-1)
        
        # Keep 2 * beam_width candidates: even if beam_width of them end in EOS,
        # beam_width live ones remain
        top_scores, top_idx = cand_scores.topk(min(2 * beam_width, cand_scores.numel()))
        
        new_beams, new_scores = [], []
        for score, idx in zip(top_scores.tolist(), top_idx.tolist()):
            beam_idx, token = divmod(idx, vocab_size)
            seq = beams[beam_idx] + [token]
            if token == eos_token:
                finished.append((seq, score))   # complete: leaves the beam
            else:
                new_beams.append(seq)
                new_scores.append(score)
                if len(new_beams) == beam_width:
                    break
        beams, scores = new_beams, torch.tensor(new_scores)
        
        # Early stopping: beam_width complete hypotheses are enough
        if len(finished) >= beam_width or not beams:
            break
    
    # Step 3: Pick the best hypothesis by length-normalised score. Live beams that
    # ran into max_len compete too (unless we stopped early on finished ones).
    candidates = list(finished)
    if len(finished) < beam_width:
        candidates += list(zip(beams, scores.tolist()))
    
    def normalised(candidate):
        seq, score = candidate
        return score / max(len(seq) - 1, 1) ** length_penalty
    
    best_seq, _ = max(candidates, key=normalised)
    return best_seq

In [ ]:
# Validation
print("=== Beam Search Validation ===")
print()

beam_width = 5
max_len = 20

# Test 1: Output sequence should have the correct length
result = beam_search(model, start_token, beam_width=beam_width, max_len=max_len)
assert isinstance(result, list), f"Output should be a list, got {type(result)}"
assert len(result) == max_len, f"Sequence length should be {max_len}, got {len(result)}"
print(f"Test 1 PASSED: Output sequence has length {len(result)}")
print(f"  Sequence: {result[:10]}... (first 10 tokens)")

# Test 2: First token should be start_token
assert result[0] == start_token, f"First token should be {start_token}, got {result[0]}"
print(f"Test 2 PASSED: First token is start_token ({start_token})")

# Test 3: All tokens should be valid vocabulary indices
assert all(0 <= t < vocab_size for t in result), "All tokens should be valid vocab indices"
print(f"Test 3 PASSED: All tokens are valid indices in [0, {vocab_size})")

# Test 4: Beam search with beam_width=1 should equal greedy decoding
greedy_result = beam_search(model, start_token, beam_width=1, max_len=10)
# Verify greedy by running manually
greedy_manual = [start_token]
for _ in range(9):
    with torch.no_grad():
        logits = model(greedy_manual[-1])
    next_token = logits.argmax(dim=-1).item()
    greedy_manual.append(next_token)
assert greedy_result == greedy_manual, \
    f"beam_width=1 should match greedy decoding.\nBeam: {greedy_result}\nGreedy: {greedy_manual}"
print(f"Test 4 PASSED: beam_width=1 matches greedy decoding")

# Test 5: Different beam widths should work
for bw in [2, 3, 10]:
    res = beam_search(model, start_token, beam_width=bw, max_len=10)
    assert len(res) == 10
    print(f"Test 5: beam_width={bw} produced sequence of length {len(res)}")

# Test 6: EOS handling and length normalisation on a tiny hand-built bigram model.
# 0 -> 1 -> EOS has the best raw sum of log-probs (-1.13) but -0.57 per token;
# 0 -> 2 -> 3 -> 4 -> 5 -> 6 -> 3 -> 4 sums to -1.94 but is only -0.28 per token.
EOS = 7
probs = torch.full((8, 8), 1e-4)
probs[0, 1], probs[0, 2] = 0.46, 0.44
probs[1, EOS] = 0.70
for a, b in [(2, 3), (3, 4), (4, 5), (5, 6), (6, 3)]:
    probs[a, b] = 0.83
for row in range(8):                      # spread the remaining mass over non-EOS tokens
    free = (probs[row] == 1e-4)
    free[EOS] = False
    probs[row, free] = (1 - probs[row, ~free].sum()) / free.sum()

class TableLM(nn.Module):
    def forward(self, token_id):
        return probs[token_id].log().unsqueeze(0)

short_seq, long_seq = [0, 1, EOS], [0, 2, 3, 4, 5, 6, 3, 4]
raw = beam_search(TableLM(), 0, beam_width=2, max_len=8, eos_token=EOS, length_penalty=0.0)
normed = beam_search(TableLM(), 0, beam_width=2, max_len=8, eos_token=EOS, length_penalty=1.0)
print(f"Test 6: length_penalty=0 -> {raw}, length_penalty=1 -> {normed}")
assert raw == short_seq, f"With raw scores the finished hypothesis {short_seq} should win, got {raw}"
assert normed == long_seq, f"With length normalisation {long_seq} should win, got {normed}"
print("Test 6 PASSED: finished hypotheses are kept, and length normalisation changes the winner")

print("\nAll tests passed!")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("beam-search", beam_search=beam_search)